In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, yaml
from functional_training import train_round
runtime = json.loads(Path(RUNTIME_PATH).read_text())
config = yaml.safe_load(Path(runtime["config"]).read_text())
print(json.dumps({"config_path": runtime["config"], "config_sha256": runtime["config_sha256"], "seeds": config["shared"]["model_seeds"], "git_sha": runtime["git_sha"], "code_sha256": runtime["code_sha256"], "run_id": runtime["run_id"], "party": runtime["party"], "run_at": datetime.now(timezone.utc).isoformat()}, ensure_ascii=False))

In [ ]:
train_round(runtime)

In [ ]:
if runtime["party"] == "alice":
    import pandas as pd
    report = json.loads((Path("/srv/vfl") / "alice" / runtime["run_id"] / "results/private_evaluation.json").read_text())
    rows = []
    for item in report["metrics"]:
        for name, value in item["metrics"].items():
            rows.append({"seed": item["seed"], "route": item["route"], "metric": name, **value})
    display(pd.DataFrame(rows))
else:
    print("Bob: no plaintext labels, gradients, scores or joint weights released")

本轮为双虚拟机功能模拟，采用已冻结预对齐切片，在安全计算内验证键与顺序一致；不等于真实客户 PSI。指标、选参和预测只在 Alice 私有输出。多种子共用同一测试切片，不代表多个人群；输出攻击诊断及限制见私有结果。